# 🏠 California Housing Fiyat Tahmini — XGBoost Regression

**🇹🇷 Türkçe:**
Bu notebook, California'daki mahalle bloklarının gelir, konut yaşı, oda sayısı, nüfus ve okyanusa yakınlık gibi özelliklerinden **medyan ev fiyatını** (`median_house_value`) tahmin ediyor.

Akış: veri temizliği (eksik değer doldurma, kategorik kodlama) → aykırı değer (outlier) analizi → dokuz farklı regresyon modelinin karşılaştırılması → en iyi performansı veren **XGBoost Regressor**'ün ayarlanması.

**🇬🇧 English:**
This notebook predicts the **median house value** (`median_house_value`) of California block groups from features such as income, housing age, room counts, population, and proximity to the ocean.

The flow: data cleaning (missing-value imputation, categorical encoding) → outlier analysis → a comparison of nine regression models → tuning the best performer, the **XGBoost Regressor**.

## 📦 0. Kütüphaneler / Imports

**🇹🇷 Türkçe:**
Veri işleme ve görselleştirme için temel kütüphaneler yükleniyor. Modelleme kütüphaneleri, ihtiyaç duyuldukları bölümde ayrıca içe aktarılıyor.

**🇬🇧 English:**
The core libraries for data handling and visualization are loaded. The modeling libraries are imported later, in the section where they are needed.

In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt


## 📥 1. Veri Setini Yükleme / Load the Dataset

**🇹🇷 Türkçe:**
California Housing veri seti okunuyor. 20.640 satır ve 10 kolon var. Önemli bir ayrıntı: her satır **tek bir ev değil, bir mahalle bloğu** (block group) temsil ediyor. Bu yüzden `total_rooms`, `population` gibi kolonlar o bloktaki toplam değerlerdir.

**🇬🇧 English:**
The California Housing dataset is loaded. It has 20,640 rows and 10 columns. An important detail: each row represents **a block group, not a single house**. That is why columns such as `total_rooms` and `population` are totals for the whole block.

In [2]:
df = pd.read_csv("../../Data/21-housing.csv")

## 🔍 2. Veriye İlk Bakış / First Look at the Data

**🇹🇷 Türkçe:**
`head()`, `info()`, `describe()`, eksik değer ve kopya satır kontrolleri yapılıyor. Kolonlar:

- **`longitude`, `latitude`** — bloğun coğrafi konumu
- **`housing_median_age`** — bloktaki konutların medyan yaşı
- **`total_rooms`, `total_bedrooms`, `population`, `households`** — bloktaki toplam oda, yatak odası, nüfus ve hane sayısı
- **`median_income`** — hane başına medyan gelir (on binlerce dolar cinsinden)
- **`ocean_proximity`** — okyanusa yakınlık kategorisi (tek metin kolonu)
- **`median_house_value`** — **hedef değişken**: medyan ev fiyatı (dolar)

Kopya satır yok, ancak `total_bedrooms` kolonunda 207 eksik değer bulunuyor.

**🇬🇧 English:**
`head()`, `info()`, `describe()`, and checks for missing values and duplicate rows are performed. The columns:

- **`longitude`, `latitude`** — the geographic location of the block
- **`housing_median_age`** — median age of the houses in the block
- **`total_rooms`, `total_bedrooms`, `population`, `households`** — block totals for rooms, bedrooms, population, and households
- **`median_income`** — median household income (in tens of thousands of dollars)
- **`ocean_proximity`** — proximity-to-ocean category (the only text column)
- **`median_house_value`** — the **target variable**: median house value in dollars

There are no duplicate rows, but `total_bedrooms` has 207 missing values.

In [3]:
df.head()

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,ocean_proximity
0,-122.23,37.88,41.0,880.0,129.0,322.0,126.0,8.3252,452600.0,NEAR BAY
1,-122.22,37.86,21.0,7099.0,1106.0,2401.0,1138.0,8.3014,358500.0,NEAR BAY
2,-122.24,37.85,52.0,1467.0,190.0,496.0,177.0,7.2574,352100.0,NEAR BAY
3,-122.25,37.85,52.0,1274.0,235.0,558.0,219.0,5.6431,341300.0,NEAR BAY
4,-122.25,37.85,52.0,1627.0,280.0,565.0,259.0,3.8462,342200.0,NEAR BAY


In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 20640 entries, 0 to 20639
Data columns (total 10 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   longitude           20640 non-null  float64
 1   latitude            20640 non-null  float64
 2   housing_median_age  20640 non-null  float64
 3   total_rooms         20640 non-null  float64
 4   total_bedrooms      20433 non-null  float64
 5   population          20640 non-null  float64
 6   households          20640 non-null  float64
 7   median_income       20640 non-null  float64
 8   median_house_value  20640 non-null  float64
 9   ocean_proximity     20640 non-null  str    
dtypes: float64(9), str(1)
memory usage: 1.6 MB


In [5]:
df.describe()

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value
count,20640.000000,20640.000000,20640.000000,20640.000000,20433.000000,20640.000000,20640.000000,20640.000000,20640.000000
mean,-119.569704,35.631861,28.639486,2635.763081,537.870553,1425.476744,499.539680,3.870671,206855.816909
std,2.003532,2.135952,12.585558,2181.615252,421.385070,1132.462122,382.329753,1.899822,115395.615874
min,-124.350000,32.540000,1.000000,2.000000,1.000000,3.000000,1.000000,0.499900,14999.000000
25%,-121.800000,33.930000,18.000000,1447.750000,296.000000,787.000000,280.000000,2.563400,119600.000000
50%,-118.490000,34.260000,29.000000,2127.000000,435.000000,1166.000000,409.000000,3.534800,179700.000000
75%,-118.010000,37.710000,37.000000,3148.000000,647.000000,1725.000000,605.000000,4.743250,264725.000000
max,-114.310000,41.950000,52.000000,39320.000000,6445.000000,35682.000000,6082.000000,15.000100,500001.000000


In [6]:
df.isnull().sum()

longitude               0
latitude                0
housing_median_age      0
total_rooms             0
total_bedrooms        207
population              0
households              0
median_income           0
median_house_value      0
ocean_proximity         0
dtype: int64

In [7]:
df.duplicated().sum()

np.int64(0)

## 🩹 3. Eksik Değerlerin Doldurulması / Handling Missing Values

**🇹🇷 Türkçe:**
`total_bedrooms` kolonundaki 207 eksik değer, kolonun **mod** (en sık görülen) değeriyle dolduruluyor ve ardından `isnull().sum()` ile kontrol ediliyor.

**Not:** Sürekli (continuous) sayısal bir kolonda genellikle **medyan** tercih edilir; mod, kesikli veya kategorik değişkenler için daha uygundur. Eksik oran %1'in altında olduğu için (207/20.640) bu tercih sonucu belirgin biçimde etkilemiyor.

**🇬🇧 English:**
The 207 missing values in `total_bedrooms` are filled with the column's **mode** (most frequent value), then verified with `isnull().sum()`.

**Note:** For a continuous numeric column the **median** is usually preferred; the mode is better suited to discrete or categorical variables. Since less than 1% of the values are missing (207 out of 20,640), this choice does not meaningfully change the results.

In [8]:
df["total_bedrooms"] = df["total_bedrooms"].fillna(df["total_bedrooms"].mode()[0])

In [9]:
df.isnull().sum()

longitude             0
latitude              0
housing_median_age    0
total_rooms           0
total_bedrooms        0
population            0
households            0
median_income         0
median_house_value    0
ocean_proximity       0
dtype: int64

## 🌊 4. `ocean_proximity` Kolonu / The `ocean_proximity` Column

**🇹🇷 Türkçe:**
Tek kategorik kolon inceleniyor ve sayısala çevriliyor. Dağılım şöyle:

| Kategori | Adet | Ortalama ev değeri |
|---|---:|---:|
| `<1H OCEAN` | 9.136 | 240.084 $ |
| `INLAND` | 6.551 | 124.805 $ |
| `NEAR OCEAN` | 2.658 | 249.434 $ |
| `NEAR BAY` | 2.290 | 259.212 $ |
| `ISLAND` | **5** | 380.440 $ |

**`ISLAND` neden `NEAR BAY` ile birleştiriliyor?** Çünkü sadece **5 satır** içeriyor — 20.640 satırlık verinin on binde ikisi. Bu kadar az örnekle ayrı bir kategori tutmak anlamsız: train/test bölmesinde bir tarafa 4, diğerine 1 satır düşer ve model bundan güvenilir hiçbir şey öğrenemez. Coğrafi olarak da en yakın kategori `NEAR BAY` olduğu için oraya aktarılıyor.

Ardından kategoriler okyanusa yakınlığa göre sıralı biçimde kodlanıyor: `INLAND → 0`, `<1H OCEAN → 1`, `NEAR OCEAN → 2`, `NEAR BAY → 3`.

**🇬🇧 English:**
The only categorical column is examined and converted to numbers. The distribution:

| Category | Count | Mean house value |
|---|---:|---:|
| `<1H OCEAN` | 9,136 | $240,084 |
| `INLAND` | 6,551 | $124,805 |
| `NEAR OCEAN` | 2,658 | $249,434 |
| `NEAR BAY` | 2,290 | $259,212 |
| `ISLAND` | **5** | $380,440 |

**Why is `ISLAND` merged into `NEAR BAY`?** Because it contains only **5 rows** — about 0.02% of the data. Keeping it as a separate category is meaningless: a train/test split would put 4 rows on one side and 1 on the other, and the model could learn nothing reliable from that. Geographically the closest category is `NEAR BAY`, so those rows are folded into it.

The categories are then encoded in order of proximity to the ocean: `INLAND → 0`, `<1H OCEAN → 1`, `NEAR OCEAN → 2`, `NEAR BAY → 3`.

In [10]:
df["ocean_proximity"].unique()

<StringArray>
['NEAR BAY', '<1H OCEAN', 'INLAND', 'NEAR OCEAN', 'ISLAND']
Length: 5, dtype: str

In [11]:
df["ocean_proximity"].value_counts()

ocean_proximity
<1H OCEAN     9136
INLAND        6551
NEAR OCEAN    2658
NEAR BAY      2290
ISLAND           5
Name: count, dtype: int64

In [12]:
df["ocean_proximity"] = df["ocean_proximity"].replace("ISLAND","NEAR BAY")

In [13]:
df["ocean_proximity"].value_counts()

ocean_proximity
<1H OCEAN     9136
INLAND        6551
NEAR OCEAN    2658
NEAR BAY      2295
Name: count, dtype: int64

In [14]:
df["ocean_proximity"] = df["ocean_proximity"].map({"INLAND":0,"<1H OCEAN":1,"NEAR OCEAN":2,"NEAR BAY":3})

In [15]:
df["ocean_proximity"].value_counts()

ocean_proximity
1    9136
0    6551
2    2658
3    2295
Name: count, dtype: int64

## 📦 5. Kutu Grafikleri ile Aykırı Değer Taraması / Boxplot Scan for Outliers

**🇹🇷 Türkçe:**
Her kolon için kutu grafiği (boxplot) çizdirilerek aykırı değerler görsel olarak taranıyor. Kutunun dışındaki noktalar, IQR kuralına göre aykırı sayılan gözlemlerdir.

**🇬🇧 English:**
A boxplot is drawn for each column to visually scan for outliers. The points outside the whiskers are the observations flagged as outliers by the IQR rule.

In [16]:
for col in df.columns:
    sns.boxplot(df[col])
    plt.show()

C:\Users\efeka\AppData\Local\Temp\ipykernel_14868\1371207353.py:3: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
C:\Users\efeka\AppData\Local\Temp\ipykernel_14868\1371207353.py:3: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
C:\Users\efeka\AppData\Local\Temp\ipykernel_14868\1371207353.py:3: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
C:\Users\efeka\AppData\Local\Temp\ipykernel_14868\1371207353.py:3: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
C:\Users\efeka\AppData\Local\Temp\ipykernel_14868\1371207353.py:3: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
C:\Users\efeka\AppData\Local\Temp\ipykernel_14868\1371207353.py:3: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
C:\Users\efeka\AppData\Local\Temp\ipykernel_14868\1371207353.py:3: UserWarni

C:\Users\efeka\AppData\Local\Temp\ipykernel_14868\1371207353.py:3: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
C:\Users\efeka\AppData\Local\Temp\ipykernel_14868\1371207353.py:3: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
C:\Users\efeka\AppData\Local\Temp\ipykernel_14868\1371207353.py:3: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 📊 6. Dağılımların İncelenmesi / Inspecting the Distributions

**🇹🇷 Türkçe:**
Dokuz sayısal kolonun histogramları tek bir 3x3 ızgarada toplanıyor. İki önemli gözlem:

1. **`total_rooms`, `total_bedrooms`, `population`, `households` sağa çarpık** — bunlar blok toplamları olduğu için büyük bloklarda doğal olarak çok yüksek değerler çıkıyor. Bu bir hata değil, verinin doğası.
2. **`median_house_value` 500.001 dolarda dik bir duvarla bitiyor** — veri toplanırken fiyatlar bu değerde **kırpılmış** (censored). 965 blok bu tavana yapışmış durumda.

**🇬🇧 English:**
Histograms of the nine numeric columns are collected into a single 3x3 grid. Two important observations:

1. **`total_rooms`, `total_bedrooms`, `population`, and `households` are right-skewed** — because they are block totals, large blocks naturally produce very high values. This is not an error but the nature of the data.
2. **`median_house_value` ends in a sharp wall at $500,001** — house values were **capped (censored)** when the data was collected. 965 blocks sit at this ceiling.

In [17]:
columns = ['longitude', 'latitude', 'housing_median_age', 'total_rooms',
       'total_bedrooms', 'population', 'households', 'median_income',
       'median_house_value']

fig, axes = plt.subplots(nrows = 3, ncols = 3, figsize=(15,12))
fig.suptitle("Distributions", fontsize = 18, fontweight = "bold")

for i, col in enumerate(columns):
    row = i // 3
    col_idx = i % 3
    ax = axes[row, col_idx]
    sns.histplot(data = df, x = col, kde=True, ax=ax, bins=30)
    ax.set_title(col, fontsize=10, fontstyle = "italic")

plt.tight_layout()
plt.show()

C:\Users\efeka\AppData\Local\Temp\ipykernel_14868\1168862423.py:16: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 🛠️ 7. Aykırı Değer Fonksiyonları / Outlier Utility Functions

**🇹🇷 Türkçe:**
IQR (Interquartile Range) yöntemiyle çalışan üç yardımcı fonksiyon tanımlanıyor:

- **`find_outliers_iqr`** — silmeden önce her kolonda kaç aykırı değer olduğunu raporlar
- **`remove_outliers_from_column`** — tek bir kolona göre aykırı satırları siler
- **`remove_outliers_from_all_columns`** — bütün sayısal kolonlar için aynı işlemi tekrarlar

**IQR kuralı:** `Q1 - 1.5 × IQR` ile `Q3 + 1.5 × IQR` aralığının dışında kalan değerler aykırı sayılır. Buradaki `1.5` katsayısı kutu grafiğinin geleneksel değeridir; büyütüldükçe (örneğin `3.0`) daha az satır aykırı olarak işaretlenir.

**🇬🇧 English:**
Three helper functions based on the IQR (Interquartile Range) method are defined:

- **`find_outliers_iqr`** — reports how many outliers each column has, before deleting anything
- **`remove_outliers_from_column`** — removes outlier rows based on a single column
- **`remove_outliers_from_all_columns`** — repeats the same operation across all numeric columns

**The IQR rule:** values outside the range `Q1 - 1.5 × IQR` to `Q3 + 1.5 × IQR` are treated as outliers. The `1.5` factor is the conventional boxplot value; raising it (to `3.0`, for example) flags fewer rows as outliers.

In [18]:
def find_outliers_iqr(df, threshold = 1.5):
    outlier_summary = {}

    numeric_cols = df.select_dtypes(include=["float64", "int64"]).columns

    for col in numeric_cols:
        Q1 = df[col].quantile(0.25)
        Q3 = df[col].quantile(0.75)
        IQR = Q3 - Q1

        lower_bound = Q1 - threshold * IQR
        upper_bound = Q3 + threshold * IQR

        outliers = df[ (df[col] < lower_bound) | (df[col] > upper_bound)]

        outlier_summary[col] = {
            "outlier_count" : outliers.shape[0],
            "outlier_percentage" : 100 * outliers.shape[0] / df.shape[0],
            "lower_bound" : lower_bound,
            "upper_bound" : upper_bound
        }
    return pd.DataFrame(outlier_summary)

In [19]:
def remove_outliers_from_column(df,target_col, threshold = 1.5):
    Q1 = df[target_col].quantile(0.25)
    Q3 = df[target_col].quantile(0.75)
    IQR = Q3 - Q1

    lower_bound = Q1 - threshold * IQR
    upper_bound = Q3 + threshold * IQR
    return df[ (df[target_col] >= lower_bound) & (df[target_col] <= upper_bound)]

In [20]:
def remove_outliers_from_all_columns(df, threshold = 1.5):
    df_clean = df.copy()
    numeric_cols = df.select_dtypes(include=["float64", "int64"]).columns

    for col in numeric_cols:
        Q1 = df[col].quantile(0.25)
        Q3 = df[col].quantile(0.75)
        IQR = Q3 - Q1

        lower_bound = Q1 - threshold * IQR
        upper_bound = Q3 + threshold * IQR

        df_clean = df_clean[(df_clean[col] >= lower_bound) & (df_clean[col] <= upper_bound)]
    return df_clean.copy()

## ⚖️ 8. İki Temizleme Stratejisinin Karşılaştırılması / Comparing Two Cleaning Strategies

**🇹🇷 Türkçe:**
İki yaklaşım karşılaştırılıyor:

| Strateji | Kalan satır | Kayıp |
|---|---:|---:|
| Ham veri | 20.640 | — |
| Sadece hedef kolonu temizle | 19.569 | %5,2 |
| Bütün kolonları temizle | 15.710 | %23,9 |

Bütün kolonlara körlemesine IQR uygulamak verinin dörtte birini siliyor — bu genellikle fazla agresiftir. Bu yüzden modelleme aşamasında **sadece hedef kolonu temizlenmiş** olan `df_target_clean` kullanılıyor.

**⚠️ Yöntemsel not:** Burada silinen 1.071 satırın **965'i**, önceki bölümde bahsedilen **500.001 dolar tavanına** yapışmış pahalı bloklardır. Bunlar hatalı ölçüm değil, kırpılmış (censored) gerçek veridir; silindiklerinde model pahalı ev aralığını hiç görmemiş olur ve raporlanan skorlar gerçekte olduğundan iyimser çıkar. Ayrıca temizlik `train_test_split`'ten **önce** uygulandığı için test setinden de satır çıkarılmış olur; daha titiz bir kurulumda önce bölme yapılıp temizlik yalnızca eğitim setine uygulanır.

**🇬🇧 English:**
Two approaches are compared:

| Strategy | Rows kept | Loss |
|---|---:|---:|
| Raw data | 20,640 | — |
| Clean the target column only | 19,569 | 5.2% |
| Clean all columns | 15,710 | 23.9% |

Applying IQR blindly to every column deletes a quarter of the data, which is usually too aggressive. So the modeling stage uses `df_target_clean`, where **only the target column** was cleaned.

**⚠️ Methodological note:** Of the 1,071 rows removed here, **965** are the expensive blocks sitting at the **$500,001 cap** mentioned earlier. These are not measurement errors but censored real data; once they are dropped, the model never sees the high-price range and the reported scores come out more optimistic than reality. Also, because the cleaning is applied **before** `train_test_split`, rows are removed from the test set as well; a stricter setup would split first and clean only the training set.

In [21]:
print("original data shape: ", df.shape)
df_target_clean = remove_outliers_from_column(df, "median_house_value")
print("only target column cleaning shape: ", df_target_clean.shape)
df_all_clean = remove_outliers_from_all_columns(df)
print("all columns cleaning shape: ", df_all_clean.shape)

original data shape:  (20640, 10)
only target column cleaning shape:  (19569, 10)
all columns cleaning shape:  (15710, 10)


## 🎯 9. Özellik/Hedef Ayrımı ve Train-Test Bölmesi / Feature-Target Split

**🇹🇷 Türkçe:**
`median_house_value` hedef değişken (`y`), geri kalan dokuz kolon özellikler (`X`) olarak ayrılıyor. Veri %70 eğitim / %30 test şeklinde bölünüyor. `random_state = 15` sabitlenerek sonuçların her çalıştırmada aynı çıkması sağlanıyor.

**🇬🇧 English:**
`median_house_value` becomes the target (`y`) and the remaining nine columns the features (`X`). The data is split 70% train / 30% test. Fixing `random_state = 15` makes the results reproducible across runs.

In [22]:
X = df_target_clean.drop("median_house_value", axis = 1)
y = df_target_clean["median_house_value"]

In [23]:
from sklearn.model_selection import train_test_split

In [24]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.3, random_state = 15)

## 🧰 10. Modeller ve Değerlendirme Fonksiyonu / Models and Evaluation Helper

**🇹🇷 Türkçe:**
Dokuz farklı regresyon modeli ve ortak bir değerlendirme fonksiyonu hazırlanıyor. `evaluate_model` fonksiyonu üç metrik döndürüyor:

- **MAE** (Mean Absolute Error) — ortalama mutlak hata; dolar cinsinden yorumlanabilir
- **RMSE** (Root Mean Squared Error) — büyük hataları daha ağır cezalandırır
- **R²** — modelin hedefteki değişimin ne kadarını açıkladığı (1'e yakın olması iyi)

**🇬🇧 English:**
Nine regression models and a shared evaluation helper are prepared. The `evaluate_model` function returns three metrics:

- **MAE** (Mean Absolute Error) — the average absolute error, interpretable in dollars
- **RMSE** (Root Mean Squared Error) — penalizes large errors more heavily
- **R²** — how much of the variation in the target the model explains (closer to 1 is better)

In [25]:
from sklearn.ensemble import RandomForestRegressor, AdaBoostRegressor, GradientBoostingRegressor
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from xgboost import XGBRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

In [26]:
def evaluate_model(true, predicted):
    mae = mean_absolute_error(true, predicted)
    mse = mean_squared_error(true, predicted)
    rmse = np.sqrt(mean_squared_error(true, predicted))
    r2_square = r2_score(true, predicted)
    return mae, rmse, r2_square

In [27]:
models = {
    "Linear Regression" : LinearRegression(),
    "Lasso" : Lasso(),
    "Ridge" : Ridge(),
    "K Neighbors Regressor" : KNeighborsRegressor(),
    "Decision Tree" : DecisionTreeRegressor(),
    "Random Forest Regressor" : RandomForestRegressor(),
    "Adaboost Regressor" : AdaBoostRegressor(),
    "Gradient Boost Regressor" : GradientBoostingRegressor(),
    "XGBoost Regressor" : XGBRegressor()
}

## 🏁 11. Modellerin Karşılaştırılması / Comparing the Models

**🇹🇷 Türkçe:**
Dokuz model sırayla eğitilip hem eğitim hem test setindeki performansları yazdırılıyor. Sonuçlar:

| Model | Eğitim R² | Test R² | Test MAE | Test RMSE |
|---|---:|---:|---:|---:|
| Linear Regression | 0.5993 | 0.6145 | 44.550 $ | 59.689 $ |
| Lasso | 0.5993 | 0.6145 | 44.550 $ | 59.689 $ |
| Ridge | 0.5993 | 0.6145 | 44.550 $ | 59.689 $ |
| K Neighbors Regressor | 0.4257 | 0.1500 | 69.869 $ | 88.635 $ |
| Decision Tree | **1.0000** | 0.5937 | 41.107 $ | 61.285 $ |
| Random Forest Regressor | 0.9705 | 0.7949 | 29.464 $ | 43.537 $ |
| Adaboost Regressor | 0.3544 | 0.3541 | 66.693 $ | 77.267 $ |
| Gradient Boost Regressor | 0.7497 | 0.7349 | 35.377 $ | 49.499 $ |
| **XGBoost Regressor** | 0.9351 | **0.8082** | **28.781 $** | **42.103 $** |

Tablodan çıkan üç ders:

1. **Decision Tree eğitimde R² = 1.0000, testte 0.5937** — ders kitabı örneği bir **aşırı öğrenme (overfitting)**. Tek bir derin ağaç eğitim verisini ezberliyor ama genelleme yapamıyor. Random Forest ve XGBoost'un varlık sebebi tam olarak bu sorunu çözmek.
2. **KNN en kötü performansı veriyor (0.1500)** — çünkü özellikler **ölçeklenmemiş**. `total_rooms` binlerle, `median_income` ise birlerle ifade ediliyor; mesafe hesabı yapan KNN'de büyük ölçekli kolon diğerlerini eziyor. KNN kullanılacaksa `StandardScaler` şart.
3. **Ağaç tabanlı topluluk (ensemble) modelleri açık ara önde** — Random Forest ve XGBoost, doğrusal modellerin yakalayamadığı doğrusal olmayan ilişkileri (özellikle konum etkisini) öğrenebiliyor.

**ℹ️ Tekrarlanabilirlik notu:** Bu sözlükteki ağaç tabanlı modellerde `random_state` sabitlenmediği için, notebook her yeniden çalıştırıldığında bu sayılar birkaç binde bir oynayabilir. Tablo, kaydedilmiş bu çalıştırmanın sonuçlarıdır.

**🇬🇧 English:**
The nine models are trained in turn and their performance on both the training and test sets is printed. Results:

| Model | Train R² | Test R² | Test MAE | Test RMSE |
|---|---:|---:|---:|---:|
| Linear Regression | 0.5993 | 0.6145 | $44,550 | $59,689 |
| Lasso | 0.5993 | 0.6145 | $44,550 | $59,689 |
| Ridge | 0.5993 | 0.6145 | $44,550 | $59,689 |
| K Neighbors Regressor | 0.4257 | 0.1500 | $69,869 | $88,635 |
| Decision Tree | **1.0000** | 0.5937 | $41,107 | $61,285 |
| Random Forest Regressor | 0.9705 | 0.7949 | $29,464 | $43,537 |
| Adaboost Regressor | 0.3544 | 0.3541 | $66,693 | $77,267 |
| Gradient Boost Regressor | 0.7497 | 0.7349 | $35,377 | $49,499 |
| **XGBoost Regressor** | 0.9351 | **0.8082** | **$28,781** | **$42,103** |

Three lessons from this table:

1. **The Decision Tree scores R² = 1.0000 on training but 0.5937 on test** — a textbook case of **overfitting**. A single deep tree memorizes the training data but fails to generalize. Solving exactly this problem is why Random Forest and XGBoost exist.
2. **KNN performs worst (0.1500)** — because the features are **not scaled**. `total_rooms` is measured in thousands while `median_income` is in single digits; in a distance-based method like KNN the large-scale column drowns out the others. Using KNN here would require `StandardScaler`.
3. **Tree-based ensembles lead by a wide margin** — Random Forest and XGBoost capture the non-linear relationships (especially the effect of location) that the linear models cannot.

**ℹ️ Reproducibility note:** The tree-based models in this dictionary are created without a fixed `random_state`, so these figures can shift slightly on every re-run. The table reflects the stored run of this notebook.

In [28]:
for i in range(len(list(models))):
    model = list(models.values())[i]
    model.fit(X_train, y_train)

    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)

    model_train_mae, model_train_rmse, model_train_r2 = evaluate_model(y_train, y_train_pred)
    model_test_mae, model_test_rmse, model_test_r2 = evaluate_model(y_test, y_test_pred)

    print(list(models.keys())[i])
    print("Model performance for Training Set")
    print("Root Mean Squared Error: ", model_train_rmse)
    print("Mean Absolute Error: ", model_train_mae)
    print("R2 Score: ", model_train_r2)

    print("-----------------------------------")

    print("Model performance for Test Set")
    print("Root Mean Squared Error: ", model_test_rmse)
    print("Mean Absolute Error: ", model_test_mae)
    print("R2 Score: ", model_test_r2)

    print("-----------------------------------")
    print("\n")

Linear Regression
Model performance for Training Set
Root Mean Squared Error:  60215.65785473253
Mean Absolute Error:  44823.214904781154
R2 Score:  0.5993423806208737
-----------------------------------
Model performance for Test Set
Root Mean Squared Error:  59688.86843401657
Mean Absolute Error:  44549.91077044777
R2 Score:  0.6145476657950735
-----------------------------------




Lasso
Model performance for Training Set
Root Mean Squared Error:  60215.657916219294
Mean Absolute Error:  44823.16437113374
R2 Score:  0.5993423798026435
-----------------------------------
Model performance for Test Set
Root Mean Squared Error:  59688.86982179469
Mean Absolute Error:  44549.903170090656
R2 Score:  0.6145476478713858
-----------------------------------


Ridge
Model performance for Training Set
Root Mean Squared Error:  60215.65834060389
Mean Absolute Error:  44823.031886901445
R2 Score:  0.599342374155178
-----------------------------------
Model performance for Test Set
Root Mean Squared Error:  59688.89901766752
Mean Absolute Error:  44549.85201789367
R2 Score:  0.6145472707953701
-----------------------------------


K Neighbors Regressor
Model performance for Training Set
Root Mean Squared Error:  72090.7503666048
Mean Absolute Error:  56399.341772521526
R2 Score:  0.4257333081832152
-----------------------------------
Model performance for Test Set
Root Mean Sq

Decision Tree
Model performance for Training Set
Root Mean Squared Error:  0.0
Mean Absolute Error:  0.0
R2 Score:  1.0
-----------------------------------
Model performance for Test Set
Root Mean Squared Error:  61284.79767856912
Mean Absolute Error:  41106.54062340317
R2 Score:  0.5936600695651
-----------------------------------




Random Forest Regressor
Model performance for Training Set
Root Mean Squared Error:  16339.104547864725
Mean Absolute Error:  10988.892634691196
R2 Score:  0.970500745628518
-----------------------------------
Model performance for Test Set
Root Mean Squared Error:  43536.72953524244
Mean Absolute Error:  29464.388165559532
R2 Score:  0.7949330876985807
-----------------------------------




Adaboost Regressor
Model performance for Training Set
Root Mean Squared Error:  76438.8703171321
Mean Absolute Error:  66458.08620557594
R2 Score:  0.3543709613677142
-----------------------------------
Model performance for Test Set
Root Mean Squared Error:  77267.15980268618
Mean Absolute Error:  66692.73991146656
R2 Score:  0.3540872075954644
-----------------------------------




Gradient Boost Regressor
Model performance for Training Set
Root Mean Squared Error:  47596.362410625465
Mean Absolute Error:  34163.02877168914
R2 Score:  0.7496762473066998
-----------------------------------
Model performance for Test Set
Root Mean Squared Error:  49498.76814746195
Mean Absolute Error:  35377.089541876485
R2 Score:  0.734922558294296
-----------------------------------




XGBoost Regressor
Model performance for Training Set
Root Mean Squared Error:  24231.33740516365
Mean Absolute Error:  17427.566070035315
R2 Score:  0.9351202294546677
-----------------------------------
Model performance for Test Set
Root Mean Squared Error:  42103.2606005206
Mean Absolute Error:  28780.933737797543
R2 Score:  0.8082146414047507
-----------------------------------




## 🚀 12. Nihai XGBoost Modeli / The Final XGBoost Model

**🇹🇷 Türkçe:**
Karşılaştırmayı kazanan XGBoost, ayarlanmış parametrelerle yeniden eğitiliyor:

- **`n_estimators = 300`** — sırayla eklenen ağaç sayısı
- **`max_depth = 6`** — her ağacın derinliği
- **`learning_rate = 0.1`** — her ağacın katkı payı; düşürülürse `n_estimators` artırılmalıdır
- **`colsample_bytree = 0.7`** — her ağaç kolonların yalnızca %70'ini görür. Random Forest'taki kolon örneklemesinin aynısı: ağaçların birbirinden farklılaşmasını sağlar ve aşırı öğrenmeyi azaltır.

**🇬🇧 English:**
XGBoost, the winner of the comparison, is retrained with tuned parameters:

- **`n_estimators = 300`** — the number of trees added sequentially
- **`max_depth = 6`** — the depth of each tree
- **`learning_rate = 0.1`** — each tree's contribution; if lowered, `n_estimators` should be raised
- **`colsample_bytree = 0.7`** — each tree sees only 70% of the columns. This is the same column sampling used in Random Forest: it decorrelates the trees and reduces overfitting.

In [29]:
model = XGBRegressor(n_estimators = 300, max_depth = 6, learning_rate = 0.1, colsample_bytree = 0.7)

In [30]:
model.fit(X_train, y_train)

,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API <callback_api>`... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,0.7
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,True
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegressor( tree_method=""hist"", eval_metric=mean_absolute_error, ) reg.fit(X, y, eval_set=[(X, y)])",None
,feature_types feature_types: typing.Optional[typing.Sequence[str]].. versionadded:: 1.7.0Used for specifying feature types without constructing a dataframe. Seethe :py:class:`DMatrix` for details.,None


In [31]:
    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)

    model_train_mae, model_train_rmse, model_train_r2 = evaluate_model(y_train, y_train_pred)
    model_test_mae, model_test_rmse, model_test_r2 = evaluate_model(y_test, y_test_pred)

    print(list(models.keys())[i])
    print("Model performance for Training Set")
    print("Root Mean Squared Error: ", model_train_rmse)
    print("Mean Absolute Error: ", model_train_mae)
    print("R2 Score: ", model_train_r2)

    print("-----------------------------------")

    print("Model performance for Test Set")
    print("Root Mean Squared Error: ", model_test_rmse)
    print("Mean Absolute Error: ", model_test_mae)
    print("R2 Score: ", model_test_r2)

    print("-----------------------------------")
    print("\n")

XGBoost Regressor
Model performance for Training Set
Root Mean Squared Error:  24727.773387416502
Mean Absolute Error:  17559.63581305209
R2 Score:  0.9324345677344236
-----------------------------------
Model performance for Test Set
Root Mean Squared Error:  41230.00956613783
Mean Absolute Error:  27951.711934189236
R2 Score:  0.8160876644384448
-----------------------------------




## ✅ 13. Sonuçlar / Results

**🇹🇷 Türkçe:**
Ayarlanmış XGBoost modelinin nihai performansı:

| | RMSE | MAE | R² |
|---|---:|---:|---:|
| **Eğitim seti** | 24.728 $ | 17.560 $ | 0.9324 |
| **Test seti** | 41.230 $ | 27.952 $ | 0.8161 |

Model, ev fiyatlarındaki değişimin **%82'sini** açıklıyor ve ortalama **27.952 dolar** yanılıyor. Bu, medyan ev fiyatının yaklaşık 177.000 dolar olduğu bir veri setinde makul bir sonuç.

Eğitim R²'si (0.9324) ile test R²'si (0.8161) arasındaki fark bir miktar aşırı öğrenmeye işaret ediyor. Bunu azaltmak için denenebilecekler: `learning_rate` düşürüp `n_estimators` artırmak, `max_depth`'i küçültmek, `subsample` ve `reg_lambda` parametrelerini eklemek, ya da `early_stopping_rounds` ile ağaç eklemeyi otomatik durdurmak.

**Geliştirme fikri:** Blok toplamı olan kolonlar yerine oran özellikleri üretmek (`total_rooms / households`, `total_bedrooms / total_rooms`, `population / households`) hem çarpıklığı azaltır hem de genellikle skoru yükseltir.

**🇬🇧 English:**
Final performance of the tuned XGBoost model:

| | RMSE | MAE | R² |
|---|---:|---:|---:|
| **Training set** | $24,728 | $17,560 | 0.9324 |
| **Test set** | $41,230 | $27,952 | 0.8161 |

The model explains **82%** of the variation in house values, with an average error of **$27,952**. That is a reasonable result for a dataset whose median house value is around $177,000.

The gap between training R² (0.9324) and test R² (0.8161) indicates some overfitting. Things worth trying to reduce it: lowering `learning_rate` while raising `n_estimators`, reducing `max_depth`, adding `subsample` and `reg_lambda`, or using `early_stopping_rounds` to stop adding trees automatically.

**Idea for improvement:** Replacing the block-total columns with ratio features (`total_rooms / households`, `total_bedrooms / total_rooms`, `population / households`) reduces skew and usually improves the score as well.